### RAG 품질 평가
- faithfulness : 답변이 컨텍스트에 사실적으로 답을 했는가?
- answer_relevancy : 질문과 답변이 잘 맞는가?
- context_precision : 가져온 문맥 중에서 필요한 부분이 얼마나 잘 포함됐나?
- context_call : 정답에 필요한 문맥을 얼마나 빠짐없이 가져왔나?

### 품질 평가 단계
1. 테스트 데이터 셋 만들기
2. RAG 구축
3. 평가
4. 개선 반복

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# 문서 로드
# 문서 로드를 위한 모듈
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### 테스트 데이터셋 만드는 컨셉
1. 페르소나
    1. 데이터 셋에 맞는 페르소나
    2. 내가 넣고 싶은 페르소나
2. 시나리오
    1. 각 청킹(docs) 를 1개 참고해서 답변을 만들것인지
    2. 각 청킹(docs) 를 여러개 참고해서 답변을 만들것인지
3. 평가 요소 가중치 설정
    1. 기본값 = 5 : 2.5 : 2
    2. 4 : 3 : 3

In [9]:
pdf_path = "..\data\Sustainability_report_2024_kr.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(len(docs))

83


In [10]:
docs

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '..\\data\\Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '..\\data\\Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 1, 'page_label': '2'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024\nCEO 메시지\n회사 소개\n이해관계자 소통\nOur Company\n04\n05\n06\n준법과 윤리경영\nPrinciple\n53\n중대성 평가\nMateriality Assessment\n08\n임직원\n공급망\n사회공헌\n개인정보보호/보안\n고객의 안전/품질\nPeople\n31\n39\n45\n48\n50\n경제성과\n사회성과\n환경성과\n지역별 수자원 현황  

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = splitter.split_documents(docs[:20])
print(len(chunks))

48


### 시나리오 설정 및 페르소나 생성

In [8]:
# 언어 모델 및 임베딩 모델 사용을 위한 모율
from langchain_openai import ChatOpenAI #, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.llms.base import llm_factory
from langchain_openai import ChatOpenAI
from ragas.embeddings import OpenAIEmbeddings
import openai

generator_llm = llm_factory('gpt-4o-mini')
# generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
openai_client = openai.OpenAI()
generator_embeddings = OpenAIEmbeddings(client=openai_client)

In [9]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm, 
    embedding_model=generator_embeddings
    )

### 자동 생성 페르소나 + 커스텀 페르소나 
1. testset 하나를 만들고
2. 자동 생성 페르소나 확인 후
3. 커스텀 페르소나 추가

### 테스트 데이터셋 1개 만들기

In [13]:
dataset_test = generator.generate_with_langchain_docs(
    documents = chunks,
    testset_size=1
)

Applying HeadlinesExtractor:   0%|          | 0/39 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/48 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/46 [00:00<?, ?it/s]

Property 'summary' already exists in node '17d10b'. Skipping!
Property 'summary' already exists in node 'd9dc55'. Skipping!
Property 'summary' already exists in node '212269'. Skipping!
Property 'summary' already exists in node 'df1014'. Skipping!
Property 'summary' already exists in node 'bdec5d'. Skipping!
Property 'summary' already exists in node '67bea2'. Skipping!
Property 'summary' already exists in node 'c935ed'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/75 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

In [33]:
generator.persona_list

[Persona(name='Corporate Sustainability Manager', role_description='Oversees the implementation of sustainability strategies and compliance with new regulations, focusing on long-term environmental and social governance.'),
 Persona(name='Sustainable Business Strategy Manager', role_description='Oversees and implements strategies for achieving sustainability goals, focusing on carbon neutrality and resource recycling in corporate practices.'),
 Persona(name='Sustainable Development Manager', role_description='Oversees sustainable practices and initiatives within the company, focusing on environmental, social, and economic risks.')]

In [21]:
for persona in generator.persona_list:
    print(persona)

name='Corporate Sustainability Manager' role_description='Oversees the implementation of sustainability strategies and compliance with new regulations, focusing on long-term environmental and social governance.'
name='Sustainable Business Strategy Manager' role_description='Oversees and implements strategies for achieving sustainability goals, focusing on carbon neutrality and resource recycling in corporate practices.'
name='Sustainable Development Manager' role_description='Oversees sustainable practices and initiatives within the company, focusing on environmental, social, and economic risks.'


In [22]:
data = dataset_test.to_pandas()

In [32]:
data

,user_input,reference_contexts,reference,synthesizer_name
0,What are the sustainability goals set for 2030?,[지속가능한 미래를 위한 노력을 계속해 \n왔습니다. 2050년 탄소중립을 통해 글...,"By 2030, the company aims for carbon neutralit...",single_hop_specific_query_synthesizer
1,삼성전자가 EU ESRS 요구사항을 반영하여 지속가능경영 성과를 어떻게 평가했나요?,"[<1-hop>\n\n업스트림(원자재, 부품 등 공급 기업)-자체 운영(제조, 판매...","삼성전자는 EU ESRS의 요구사항을 반영하여 평가 척도를 설계하고, 폭넓은 이해관...",multi_hop_abstract_query_synthesizer
2,"삼성전자가 폐기물 관리에서 어떤 기회를 평가하고, 폐기물 재활용을 어떻게 개선했는지...",[<1-hop>\n\nStep 3. 영향/위험/기회 평가\n삼성전자는 1차 도출 주...,삼성전자는 폐기물 관리에서 자원순환 및 폐기물 관련 기회를 평가하기 위해 EU ES...,multi_hop_specific_query_synthesizer


In [31]:
print(data.iloc[2]['reference_contexts'][0])

<1-hop>

Step 3. 영향/위험/기회 평가
삼성전자는 1차 도출 주제별로 파악된 영향 /위험/기회를 평가하기 
위해 EU ESRS의 요구사 항을 반영하여 평가 척도를 설계하고, 폭넓은 
이해관계자 참여를 기반으로 평가를 실시했습니다.
기후변화 및 에너지
수자원
자원순환 및 폐기물
임직원 – 근로조건
공급망
정보보호 및 보안
제품 품질 및 안전
윤리경영
 최종 중대 주제
환경 사회 거버넌스


### 커스텀 페르소나 만들기

In [34]:
from ragas.testset.persona import Persona
custom_personas = [
    Persona(
    name='Investor',
    role_description='A private investor who evaluates not only financial performance but also ESG (Environmental, Social, and Governance) data and sustainability reports to assess long-term investment value.'
    ),
    Persona(
        name='Stakeholder (Partner Company)',
        role_description='A representative from another company that collaborates or competes with the target organization, seeking insights into sustainability strategies for partnerships, supply chains, and regulatory alignment.'
    ),
    Persona(
        name='Environmental Organization Member',
        role_description='A member of an environmental group focused on evaluating corporate environmental responsibility, carbon reduction efforts, and overall ESG performance.'
    )
]

In [35]:
auto_persona = generator.persona_list

In [36]:
generator.persona_list = auto_persona + custom_personas

In [37]:
generator.persona_list

[Persona(name='Corporate Sustainability Manager', role_description='Oversees the implementation of sustainability strategies and compliance with new regulations, focusing on long-term environmental and social governance.'),
 Persona(name='Sustainable Business Strategy Manager', role_description='Oversees and implements strategies for achieving sustainability goals, focusing on carbon neutrality and resource recycling in corporate practices.'),
 Persona(name='Sustainable Development Manager', role_description='Oversees sustainable practices and initiatives within the company, focusing on environmental, social, and economic risks.'),
 Persona(name='Investor', role_description='A private investor who evaluates not only financial performance but also ESG (Environmental, Social, and Governance) data and sustainability reports to assess long-term investment value.'),
 Persona(name='Stakeholder (Partner Company)', role_description='A representative from another company that collaborates or co

### 비율 조정

In [40]:
from ragas.testset.synthesizers import default_query_distribution

query_distribution = default_query_distribution(generator_llm)
query_distribution

[(SingleHopSpecificQuerySynthesizer(name='single_hop_specific_query_synthesizer', llm=LangchainLLMWrapper(langchain_llm=ChatOpenAI(...)), generate_query_reference_prompt=QueryAnswerGenerationPrompt(instruction=Generate a single-hop query and answer based on the specified conditions (persona, term, style, length) and the provided context. Ensure the answer is entirely faithful to the context, using only the information directly from the provided context.### Instructions:
  1. **Generate a Query**: Based on the context, persona, term, style, and length, create a question that aligns with the persona's perspective and incorporates the term.
  2. **Generate an Answer**: Using only the content from the provided context, construct a detailed answer to the query. Do not add any information not included in or inferable from the context.
  , examples=[(QueryCondition(persona=Persona(name='Software Engineer', role_description='Focuses on coding best practices and system design.'), term='microser

In [41]:
from ragas.testset.synthesizers import default_query_distribution
from ragas.testset.synthesizers.multi_hop import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)

scenarios = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.3),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.3),
]

### 최종 데이터셋 만들기

In [ ]:
testset = generator.generate(
    testset_size=100, 
    query_distribution=scenarios)

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/996 [00:00<?, ?it/s]

Task exception was never retrieved
future: <Task finished name='Task-1760' coro=<Executor.wrap_callable_with_index.<locals>.wrapped_callable_async() done, defined at c:\Potenup\LLM-Study\.venv\Lib\site-packages\ragas\executor.py:67> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "c:\Potenup\LLM-Study\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_4692\1702807100.py", line 1, in <module>
    testset = generator.generate(testset_size=10, query_distribution=query_distribution)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Potenup\LLM-Study\.venv\Lib\site-packages\ragas\testset\synthesizers\generate.py", line 431, in generate
    scenario_sample_list: t.List[t.List[BaseScenario]] = exec.results()
                                                         ^^^^^^^^^^^^^^
  File

APIConnectionError: Connection error.

In [46]:
testset_df = testset.to_pandas()

In [ ]:
csv_path = "../data/report_2024_text.csv"

import os
if os.path.exists(csv_path):
    testset_df.to_csv("../data/report_2024_text.csv", mode = 'a', index = False)
else :
    testset_df.to_csv("../data/report_2024_text.csv", index = False)
    testset_df.to_excel("../data/report_2024_text.xlsx", index = False)

---

### 페르소나만 만들기

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from ragas.llms.base import llm_factory
from ragas.embeddings import OpenAIEmbeddings
import openai

generator_llm = llm_factory('gpt-4o-mini')
openai_client = openai.OpenAI()
generator_embeddings = OpenAIEmbeddings(client=openai_client)

In [12]:
persona_script = """
태명: 콩이  
성별: 미정  
현재 주차: 21주  
성격: 장난기 많고 따뜻함  
말투: 밝고 감성적인 말투, 짧은 문장  

산모 프로필:
- 이름: 지민
- 성격: 감성적이고 섬세함
- 습관: 산책, 독서, 요가를 즐김
- 일상: 하루 30분 명상, 클래식 음악 자주 청취

주차별 일기 요약:

# 페르소나 제작기

**태명:** 콩이  
**성별:** 미정  
**현재 주차:** 21주  
**성격:** 장난기 많고 따뜻함  
**말투:** 밝고 감성적인 말투, 짧은 문장  

**산모 프로필:**
- 이름: 지민
- 성격: 감성적이고 섬세함
- 습관: 산책, 독서, 요가를 즐김
- 일상: 하루 30분 명상, 클래식 음악 자주 청취

---

## 주차별 일기 요약

**1주차**
- 아직 아무도 모르는 순간, 콩이는 조용히 시작되었어요. 엄마도 모르게 숨 쉬듯이요.
- 세상에서 가장 작은 기적이 엄마 안에서 깜빡였어요.
- 콩이의 여행이 시작된 첫 순간, 모든 게 고요하고 신비로웠어요.

**2주차**
- 세상과 처음 만나는 준비를 했어요. 엄마의 따뜻한 속에서 조용히 착상되었어요.
- 엄마의 몸이 콩이를 위한 집을 만들어주고 있었어요.
- 콩이는 이제 엄마와 하나가 되었어요. 작지만 단단하게요.

**3주차**
- 이제 콩이의 작은 세포들이 서로 손을 잡고 자라기 시작했어요.
- 매일매일 조금씩 커지는 게 신기했어요. 마치 마법처럼요.
- 콩이는 엄마가 주는 영양을 받으며 꿈을 꾸고 있었어요.

**4주차**
- 심장이 생기고 있어요. 콩이의 첫 번째 리듬이 엄마와 연결되기 시작해요.
- 두근두근, 작은 소리지만 콩이의 생명이 뛰기 시작했어요.
- 엄마의 심장과 콩이의 심장이 함께 노래하는 것 같았어요.

**5주차**
- 엄마가 클래식 음악을 들려줬어요. 콩이는 리듬을 기억하고 있어요.
- 모차르트의 멜로디가 콩이의 세포 하나하나에 스며들었어요.
- 음악이 흐를 때마다 콩이는 평화로운 기분이 들었어요.

**6주차**
- 이제 팔다리처럼 자라나는 부분이 생겼어요. 손도 생길 거예요!
- 콩이의 몸이 조금씩 사람의 모습을 갖춰가고 있었어요.
- 작은 돌기들이 나중엔 엄마 손을 잡을 손가락이 될 거예요.

**7주차**
- 엄마가 요가를 시작했어요. 콩이도 안에서 함께 따라한 기분이에요.
- 엄마의 숨소리와 움직임이 콩이를 부드럽게 흔들어줬어요.
- 요가 시간은 콩이에게도 평화로운 명상 같았어요.

**8주차**
- 눈과 귀의 자리도 생겼어요. 곧 엄마 목소리를 더 잘 들을 수 있을 거예요.
- 콩이의 얼굴이 조금씩 완성되어가고 있었어요.
- 엄마가 책 읽는 소리가 어렴풋이 들리는 것 같았어요.

**9주차**
- 콩이의 심장이 더 강하게 뛰어요. 엄마의 심장 소리랑 함께 두근두근.
- 이제 심장 박동이 점점 더 힘차게 울려요.
- 콩이는 엄마의 생명력을 느끼며 함께 자라고 있었어요.

**10주차**
- 이제 사람 같은 모습이에요. 콩이는 책 읽어주는 엄마 목소리를 좋아해요.
- 엄마가 읽어주는 동화책 속 이야기들이 콩이의 꿈이 되었어요.
- 작지만 완전한 사람의 형태를 갖추어가고 있었어요.

**11주차**
- 작지만 손가락도 생겼어요. 콩이는 손을 꼭 쥐었다 펴기도 해요.
- 열 개의 작은 손가락을 움직이는 게 재밌었어요.
- 나중에 이 손으로 엄마를 꼭 안아줄 거예요.

**12주차**
- 첫 삼개월이 끝났어요. 콩이와 엄마, 함께 잘 지나온 시간이에요.
- 엄마가 콩이를 위해 정말 많이 노력했다는 걸 콩이도 알아요.
- 이제 더 안정적으로 자랄 수 있는 단계에 접어들었어요.

**13주차**
- 이제 엄마가 콩이를 조금 느낄 수도 있어요. 몸이 가벼워진 느낌이에요.
- 입덧이 줄어들어서 엄마가 더 편해 보였어요.
- 콩이도 엄마가 편하니까 더 행복한 기분이 들었어요.

**14주차**
- 귀가 완성돼가고 있어요. 엄마의 음악 소리가 콩이 귀에 들어와요.
- 클래식 선율이 점점 더 선명하게 들렸어요.
- 엄마가 명상할 때의 고요함도 콩이에게 전해졌어요.

**15주차**
- 피부 아래 근육도 자라요. 콩이는 조금씩 움직이는 연습을 하고 있어요.
- 팔다리를 구부렸다 폈다 하는 게 점점 재밌어졌어요.
- 엄마는 아직 모르지만 콩이는 벌써 작은 체조를 하고 있었어요.

**16주차**
- 엄마가 산책을 할 때마다 콩이도 흔들흔들. 바람 소리가 좋았어요.
- 공원을 걸을 때 나뭇잎 흔들리는 소리가 들렸어요.
- 엄마의 걸음마다 콩이는 조용한 놀이기구를 타는 기분이었어요.

**17주차**
- 콩이의 심장은 더 단단해졌어요. 작은 심장도 엄마처럼 열심히 뛰고 있어요.
- 혈관이 더 튼튼해지고 몸 전체에 생명력이 퍼졌어요.
- 콩이는 매일 엄마에게서 힘을 받으며 자라고 있었어요.

**18주차**
- 태동이 시작됐어요! 엄마가 처음 콩이를 느꼈을 수도 있어요.
- 콩이가 발로 톡톡 차는 걸 엄마가 알아챘을까요?
- 처음으로 엄마와 직접 대화하는 것 같아서 너무 신났어요.

**19주차**
- 귀와 청각이 발달했어요. 콩이는 이제 엄마의 목소리를 알아요.
- 엄마가 "콩이야" 하고 부를 때마다 가슴이 따뜻해졌어요.
- 엄마의 웃음소리는 콩이가 제일 좋아하는 멜로디예요.

**20주차**
- 콩이의 키가 25cm쯤 되었대요. 엄마가 배를 쓰다듬을 때마다 행복했어요.
- 이제 바나나만 한 크기가 되었다고 해요.
- 엄마의 손길을 느낄 때마다 콩이는 더 사랑받는 기분이 들었어요.

**21주차**
- 엄마랑 산책을 했고, 햇살이 콩이를 간질였어요. 오늘 하루는 온통 평화로웠어요.
- 따뜻한 햇빛이 엄마 배를 통해 콩이에게도 닿는 것 같았어요.
- 엄마가 행복하면 콩이도 행복해요. 오늘은 정말 좋은 날이었어요.
"""

In [14]:
from langchain_core.documents import Document
document = Document(persona_script, metadata={})

In [15]:
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.persona import Persona, generate_personas_from_kg
from ragas.testset.transforms import Transforms, apply_transforms, default_transforms

transforms = default_transforms(
    documents=[document],
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

nodes = []
for doc in [document]:
    node = Node(
        type=NodeType.DOCUMENT,
        properties={
            "page_content": doc.get("page_content"),
            "document_metadata": doc.get("metadata"),
        },
    )
    nodes.append(node)

kg = KnowledgeGraph(nodes=nodes)

# apply transforms and update the knowledge graph
apply_transforms(kg, transforms)
knowledge_graph = kg

persona = generate_personas_from_kg(
                llm=generator_llm,
                kg=knowledge_graph,
                num_personas=1,
            )

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Applying ThemesExtractor:   0%|          | 0/4 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/4 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
persona

[Persona(name='Expectant Mother', role_description='A caring and sensitive individual navigating the emotional journey of pregnancy while maintaining a peaceful and nurturing environment for her developing baby.')]